In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/train.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/sampleSubmission.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/stores.csv
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/features.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/test.csv.zip


In [4]:
!pip install mlflow
!pip install dagshub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 56.4 MB/s eta 0:00:0000:01m:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 100.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.6/954.6 kB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.9/214.9 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
%pip install -q neuralforecast mlflow dagshub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 8.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.6/348.6 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.8/449.8 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 7.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires tornado==6.5.1, but you have tornado 6.5.7 which is incompatible.
Note: you may need to restart the kernel to 

In [8]:
import os
import numpy as np
import pandas as pd
import mlflow

from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS
from neuralforecast.losses.pytorch import MAE

pd.set_option("display.max_columns", 50)
np.random.seed(42)

In [9]:
import dagshub
dagshub.init(repo_owner='mesata', repo_name='Walmart---Store-Sales-Forecasting', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=139f161d-2f42-458a-a70b-6f9e7b973af9&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=910e2001392d41ee28a5e55315cf0a531df7164330a336a3d159ba3a0e41d39b




Accessing as mkekn23

Initialized MLflow to track repo "mesata/Walmart---Store-Sales-Forecasting"

Repository mesata/Walmart---Store-Sales-Forecasting initialized!

In [11]:
import mlflow

In [12]:
TRAIN_PATH = '/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/train.csv.zip'
TEST_PATH = '/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/test.csv.zip'
FEATURES_PATH ='/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/features.csv.zip'
STORES_PATH = '/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/stores.csv'

VAL_WEEKS = 8
HORIZON = VAL_WEEKS
INPUT_SIZE = 52
MIN_SERIES_LENGTH = INPUT_SIZE + HORIZON + 4

# --- DagsHub / MLflow ---
DAGSHUB_REPO_OWNER = "mesata"
DAGSHUB_REPO_NAME = "Walmart---Store-Sales-Forecasting"
DAGSHUB_TRACKING_URI = f"https://dagshub.com/{DAGSHUB_REPO_OWNER}/{DAGSHUB_REPO_NAME}.mlflow"

os.environ.setdefault("MLFLOW_TRACKING_URI", DAGSHUB_TRACKING_URI)
# os.environ.setdefault("MLFLOW_TRACKING_USERNAME", "your-username")
# os.environ.setdefault("MLFLOW_TRACKING_PASSWORD", "your-dagshub-token")

MLFLOW_EXPERIMENT = "N-BEATS"
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment(MLFLOW_EXPERIMENT)

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

MLflow tracking URI: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow


In [13]:
train_raw = pd.read_csv(TRAIN_PATH, parse_dates=["Date"])

nf_df = train_raw.copy()
nf_df["unique_id"] = nf_df["Store"].astype(str) + "_" + nf_df["Dept"].astype(str)
nf_df = nf_df.rename(columns={"Date": "ds", "Weekly_Sales": "y"})
nf_df["is_holiday_flag"] = nf_df["IsHoliday"].astype(int)

series_lengths = nf_df.groupby("unique_id")["ds"].count()
valid_ids = series_lengths[series_lengths >= MIN_SERIES_LENGTH].index
nf_df = nf_df[nf_df["unique_id"].isin(valid_ids)].reset_index(drop=True)

nf_df = nf_df[["unique_id", "ds", "y", "is_holiday_flag"]].sort_values(["unique_id", "ds"]).reset_index(drop=True)

print(f"სერიების რაოდენობა: {nf_df['unique_id'].nunique()}")
nf_df.head()

სერიების რაოდენობა: 2967


,unique_id,ds,y,is_holiday_flag
0,10_1,2010-02-05,40212.84,0
1,10_1,2010-02-12,67699.32,1
2,10_1,2010-02-19,49748.33,0
3,10_1,2010-02-26,33601.22,0
4,10_1,2010-03-05,36572.44,0


In [14]:
def wmae(y_true: np.ndarray, y_pred: np.ndarray, is_holiday: np.ndarray) -> float:
    weights = np.where(is_holiday, 5.0, 1.0)
    return float(np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights))


max_date = nf_df["ds"].max()
val_cutoff = max_date - pd.Timedelta(weeks=VAL_WEEKS)

train_df = nf_df[nf_df["ds"] <= val_cutoff].copy()
val_df = nf_df[nf_df["ds"] > val_cutoff].copy()

common_ids = set(train_df["unique_id"]) & set(val_df["unique_id"])
train_df = train_df[train_df["unique_id"].isin(common_ids)].reset_index(drop=True)
val_df = val_df[val_df["unique_id"].isin(common_ids)].reset_index(drop=True)

print(f"Train: {len(train_df)} rows, Val: {len(val_df)} rows, სერიები: {len(common_ids)}")

Train: 390346 rows, Val: 23317 rows, სერიები: 2947


In [15]:
def run_pipeline(run_name: str, model_params: dict, val_size: int = 0, notes: str = ""):
    feature_cols = ["unique_id", "ds", "y"]
    input_size = model_params["input_size"]
    min_len_needed = input_size + val_size

    train_counts = train_df.groupby("unique_id")["ds"].count()
    ok_ids = train_counts[train_counts >= min_len_needed].index
    n_dropped = train_df["unique_id"].nunique() - len(ok_ids)

    train_sub = train_df[train_df["unique_id"].isin(ok_ids)]
    val_sub = val_df[val_df["unique_id"].isin(ok_ids)]

    with mlflow.start_run(run_name=run_name):
        log_params = {k: v for k, v in model_params.items() if k != "loss"}
        log_params["loss"] = type(model_params["loss"]).__name__
        mlflow.log_params(log_params)
        mlflow.log_param("model_type", "NBEATS")
        mlflow.set_tag("notes", notes)
        mlflow.log_param("val_weeks", VAL_WEEKS)
        mlflow.log_param("n_series_train", len(ok_ids))
        mlflow.log_param("n_series_dropped_too_short", n_dropped)

        model = NBEATS(**model_params)
        nf_run = NeuralForecast(models=[model], freq="W-FRI")

        fit_kwargs = {"df": train_sub[feature_cols]}
        if val_size > 0:
            fit_kwargs["val_size"] = val_size
        nf_run.fit(**fit_kwargs)

        val_fc = nf_run.predict().rename(columns={"NBEATS": "y_pred"})
        val_ev = val_sub.merge(val_fc, on=["unique_id", "ds"], how="inner")
        v_wmae = wmae(val_ev["y"].values, val_ev["y_pred"].values, val_ev["is_holiday_flag"].values.astype(bool))

        train_cv_run = nf_run.cross_validation(
            df=train_sub[feature_cols], n_windows=1, step_size=model_params["h"]
        ).rename(columns={"NBEATS": "y_pred"})
        train_cv_run = train_cv_run.merge(
            nf_df[["unique_id", "ds", "is_holiday_flag"]], on=["unique_id", "ds"], how="left"
        )
        t_wmae = wmae(
            train_cv_run["y"].values, train_cv_run["y_pred"].values,
            train_cv_run["is_holiday_flag"].values.astype(bool)
        )

        mlflow.log_metric("train_wmae", t_wmae)
        mlflow.log_metric("val_wmae", v_wmae)

        print(f"[{run_name}] train_wmae={t_wmae:,.2f}  val_wmae={v_wmae:,.2f}  "
              f"(series: {len(ok_ids)}, dropped: {n_dropped})")
        return t_wmae, v_wmae, nf_run

In [16]:
from neuralforecast.losses.pytorch import MAE

In [19]:
nbeats_baseline_params = dict(
    h=HORIZON,
    input_size=INPUT_SIZE,
    max_steps=200,
    batch_size=32,
    learning_rate=1e-3,
    loss=MAE(),
    random_seed=42,
    scaler_type="standard",
    devices=1,          # <-- ახალი: multi-GPU DDP-ს ვბლოკავთ
    accelerator="gpu",  # <-- ცალსახად ვუთითებთ, ერთი GPU-ს გამოსაყენებლად
)

nbeats_train_wmae, nbeats_val_wmae, nf_nbeats = run_pipeline(
    run_name="nbeats_baseline",
    model_params=nbeats_baseline_params,
    val_size=0,
    notes="NBEATS baseline: input_size=52, max_steps=200, single-GPU (devices=1), exogenous features გარეშე",
)

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name                | Type          | Params | Mode 
--------------------------------------------------------------
0 | loss                | MAE           | 0      | train
1 | hist_cat_embeddings | ModuleList    | 0      | train
2 | futr_cat_embeddings | ModuleList    | 0      | train
3 | stat_cat_embeddings | ModuleList    | 0      | train
4 | padder_train        | ConstantPad1d | 0      | train
5 | scaler              | TemporalNorm  | 0      | train
6 | blocks              | ModuleList    | 2.5 M  | train
--------------------------------------------------------------
2.5 M     Trainable params
1.0 K     Non-trainable params
2.5 M     Total params
9.978     Total estimated model params size (MB)
34        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=200` reached.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting: |          | 0/? [00:00<?, ?it/s]

[nbeats_baseline] train_wmae=1,275.53  val_wmae=1,344.48  (series: 2947, dropped: 0)
🏃 View run nbeats_baseline at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/3/runs/f15796e8c7684d62923f06396a9cf0c6
🧪 View experiment at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/3


In [20]:
with mlflow.start_run(run_name="naive_baselines_nbeats_context"):
    mlflow.set_tag("notes", "იგივე naive baselines, NBEATS run-ებთან ერთად შესადარებლად")

    last_vals = train_df.sort_values("ds").groupby("unique_id")["y"].last()
    naive_last_pred = val_df["unique_id"].map(last_vals)
    naive_last_wmae = wmae(
        val_df["y"].values, naive_last_pred.values, val_df["is_holiday_flag"].values.astype(bool)
    )

    lookup = nf_df.set_index(["unique_id", "ds"])["y"]
    seasonal_ds = val_df["ds"] - pd.Timedelta(weeks=52)
    seasonal_keys = list(zip(val_df["unique_id"], seasonal_ds))
    naive_seasonal_pred = pd.Series([lookup.get(k, np.nan) for k in seasonal_keys], index=val_df.index)
    mask = naive_seasonal_pred.notna()
    naive_seasonal_wmae = wmae(
        val_df.loc[mask, "y"].values, naive_seasonal_pred[mask].values,
        val_df.loc[mask, "is_holiday_flag"].values.astype(bool)
    )

    mlflow.log_metric("naive_last_val_wmae", naive_last_wmae)
    mlflow.log_metric("naive_seasonal_val_wmae", naive_seasonal_wmae)

    print(f"Naive (last value)     val_wmae = {naive_last_wmae:,.2f}")
    print(f"Naive (seasonal t-52)  val_wmae = {naive_seasonal_wmae:,.2f}")
    print(f"NBEATS baseline        val_wmae = {nbeats_val_wmae:,.2f}")

Naive (last value)     val_wmae = 2,415.07
Naive (seasonal t-52)  val_wmae = 1,718.32
NBEATS baseline        val_wmae = 1,344.48
🏃 View run naive_baselines_nbeats_context at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/3/runs/b44ca6aff55b4b0fb06fe2140c58672e
🧪 View experiment at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/3


In [21]:
configs_nbeats = {
    "nbeats_steps_600": dict(
        h=HORIZON, input_size=INPUT_SIZE,
        max_steps=600,
        batch_size=32, learning_rate=1e-3,
        loss=MAE(), random_seed=42, scaler_type="standard",
        devices=1, accelerator="gpu",
    ),
    "nbeats_longer_input": dict(
        h=HORIZON, input_size=104,
        max_steps=600,
        batch_size=32, learning_rate=1e-3,
        loss=MAE(), random_seed=42, scaler_type="standard",
        devices=1, accelerator="gpu",
    ),
    "nbeats_interpretable": dict(
        h=HORIZON, input_size=104,
        max_steps=600,
        stack_types=["trend", "seasonality"],   # <-- generic-ის ნაცვლად, interpretable decomposition
        n_blocks=[3, 3],
        batch_size=32, learning_rate=1e-3,
        loss=MAE(), random_seed=42, scaler_type="standard",
        devices=1, accelerator="gpu",
    ),
    "nbeats_more_steps": dict(
        h=HORIZON, input_size=104,
        max_steps=1000,
        batch_size=32, learning_rate=5e-4,
        loss=MAE(), random_seed=42, scaler_type="standard",
        devices=1, accelerator="gpu",
    ),
}

results_nbeats = {}
for run_name, params in configs_nbeats.items():
    t_wmae, v_wmae, _ = run_pipeline(
        run_name=run_name,
        model_params=params,
        val_size=0,
        notes=f"NBEATS sweep: input_size={params['input_size']}, max_steps={params['max_steps']}, "
              f"stack_types={params.get('stack_types', 'generic (default)')}",
    )
    results_nbeats[run_name] = (t_wmae, v_wmae)

print("\n=== NBEATS შედარება ===")
print(f"{'run':<28}{'train_wmae':>12}{'val_wmae':>12}")
print(f"{'nbeats_baseline':<28}{nbeats_train_wmae:>12,.2f}{nbeats_val_wmae:>12,.2f}")
for run_name, (t, v) in results_nbeats.items():
    print(f"{run_name:<28}{t:>12,.2f}{v:>12,.2f}")

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name                | Type          | Params | Mode 
--------------------------------------------------------------
0 | loss                | MAE           | 0      | train
1 | hist_cat_embeddings | ModuleList    | 0      | train
2 | futr_cat_embeddings | ModuleList    | 0      | train
3 | stat_cat_embeddings | ModuleList    | 0      | train
4 | padder_train        | ConstantPad1d | 0      | train
5 | scaler              | TemporalNorm  | 0      | train
6 | blocks              | ModuleList    | 2.5 M  | train
--------------------------------------------------------------
2.5 M     Trainable params
1.0 K     Non-trainable params
2.5 M     Total params
9.978     Total estimated model params size (MB)
34        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=1000` reached.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting: |          | 0/? [00:00<?, ?it/s]

[nbeats_more_steps] train_wmae=1,187.58  val_wmae=1,402.94  (series: 2846, dropped: 101)
🏃 View run nbeats_more_steps at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/3/runs/c15fed90ecf1482abba30a8cc8247d21
🧪 View experiment at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/3

=== NBEATS შედარება ===
run                           train_wmae    val_wmae
nbeats_baseline                 1,275.53    1,344.48
nbeats_steps_600                1,271.45    1,379.37
nbeats_longer_input             1,206.22    1,349.53
nbeats_interpretable            1,218.55    1,389.03
nbeats_more_steps               1,187.58    1,402.94


In [23]:
configs_nbeats_fine = {
    "nbeats_steps_100": dict(
        h=HORIZON, input_size=INPUT_SIZE, max_steps=100,
        batch_size=32, learning_rate=1e-3,
        loss=MAE(), random_seed=42, scaler_type="standard",
        devices=1, accelerator="gpu",
    ),
    "nbeats_steps_150": dict(
        h=HORIZON, input_size=INPUT_SIZE, max_steps=150,
        batch_size=32, learning_rate=1e-3,
        loss=MAE(), random_seed=42, scaler_type="standard",
        devices=1, accelerator="gpu",
    ),
    "nbeats_steps_250": dict(
        h=HORIZON, input_size=INPUT_SIZE, max_steps=250,
        batch_size=32, learning_rate=1e-3,
        loss=MAE(), random_seed=42, scaler_type="standard",
        devices=1, accelerator="gpu",
    ),
}

results_fine = {}
for run_name, params in configs_nbeats_fine.items():
    t_wmae, v_wmae, _ = run_pipeline(
        run_name=run_name, model_params=params, val_size=0,
        notes=f"fine sweep around baseline optimum: max_steps={params['max_steps']}",
    )
    results_fine[run_name] = (t_wmae, v_wmae)

print(f"{'nbeats_baseline (200)':<28}{nbeats_train_wmae:>12,.2f}{nbeats_val_wmae:>12,.2f}")
for name, (t, v) in results_fine.items():
    print(f"{name:<28}{t:>12,.2f}{v:>12,.2f}")

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name                | Type          | Params | Mode 
--------------------------------------------------------------
0 | loss                | MAE           | 0      | train
1 | hist_cat_embeddings | ModuleList    | 0      | train
2 | futr_cat_embeddings | ModuleList    | 0      | train
3 | stat_cat_embeddings | ModuleList    | 0      | train
4 | padder_train        | ConstantPad1d | 0      | train
5 | scaler              | TemporalNorm  | 0      | train
6 | blocks              | ModuleList    | 2.5 M  | train
--------------------------------------------------------------
2.5 M     Trainable params
1.0 K     Non-trainable params
2.5 M     Total params
9.978     Total estimated model params size (MB)
34        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=250` reached.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting: |          | 0/? [00:00<?, ?it/s]

[nbeats_steps_250] train_wmae=1,267.03  val_wmae=1,352.90  (series: 2947, dropped: 0)
🏃 View run nbeats_steps_250 at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/3/runs/f1bbbab5623d4635a6838897cc1ce943
🧪 View experiment at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/3
nbeats_baseline (200)           1,275.53    1,344.48
nbeats_steps_100                1,421.96    1,422.68
nbeats_steps_150                1,288.38    1,345.82
nbeats_steps_250                1,267.03    1,352.90


In [26]:
configs_nbeats_reg = {
    "nbeats_weight_decay": dict(
        h=HORIZON, input_size=INPUT_SIZE, max_steps=300,
        batch_size=32, learning_rate=1e-3,
        loss=MAE(), random_seed=42, scaler_type="standard",
        devices=1, accelerator="gpu",
        optimizer_kwargs={"weight_decay": 1e-4},
    ),
}

results_reg = {}
for run_name, params in configs_nbeats_reg.items():
    try:
        t_wmae, v_wmae, _ = run_pipeline(
            run_name=run_name, model_params=params, val_size=0,
            notes=f"regularization sweep: weight_decay={params.get('optimizer_kwargs', {}).get('weight_decay', 0)}, "
                  f"max_steps={params['max_steps']}",
        )
        results_reg[run_name] = (t_wmae, v_wmae)
    except Exception as e:
        print(f"[{run_name}] FAILED: {e}")

for name, (t, v) in results_reg.items():
    print(f"{name:<28}{t:>12,.2f}{v:>12,.2f}")

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/neuralforecast/common/_base_model.py:801: UserWarning: ignoring optimizer_kwargs as the optimizer is not specified
  warnings.warn(

  | Name                | Type          | Params | Mode 
--------------------------------------------------------------
0 | loss                | MAE           | 0      | train
1 | hist_cat_embeddings | ModuleList    | 0      | train
2 | futr_cat_embeddings | ModuleList    | 0      | train
3 | stat_cat_embeddings | ModuleList    | 0      | train
4 | padder_train        | ConstantPad1d | 0      | train
5 | scaler              | TemporalNorm  | 0      | train
6 | blocks              | ModuleList    | 2.5 M  | train
--------------------------------------------------------------
2.5 M     Trainable params
1.0 K     Non-trainable params
2.5 M     Total params
9.978     Total estimated

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=300` reached.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting: |          | 0/? [00:00<?, ?it/s]

[nbeats_weight_decay] train_wmae=1,259.34  val_wmae=1,374.83  (series: 2947, dropped: 0)
🏃 View run nbeats_weight_decay at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/3/runs/ea68c2620acd4a1db734c664b9bd6719
🧪 View experiment at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/3
nbeats_weight_decay             1,259.34    1,374.83


In [30]:
def run_pipeline(run_name: str, model_params: dict, val_size: int = 0, notes: str = ""):
    feature_cols = ["unique_id", "ds", "y"]
    input_size = model_params["input_size"]
    min_len_needed = input_size + val_size
    train_counts = train_df.groupby("unique_id")["ds"].count()
    ok_ids = train_counts[train_counts >= min_len_needed].index
    n_dropped = train_df["unique_id"].nunique() - len(ok_ids)
    train_sub = train_df[train_df["unique_id"].isin(ok_ids)]
    val_sub = val_df[val_df["unique_id"].isin(ok_ids)]
    with mlflow.start_run(run_name=run_name):
        log_params = {k: v for k, v in model_params.items() if k != "loss"}
        log_params["loss"] = type(model_params["loss"]).__name__
        mlflow.log_params(log_params)
        mlflow.log_param("model_type", "NBEATS")
        mlflow.set_tag("notes", notes)
        mlflow.log_param("val_weeks", VAL_WEEKS)
        mlflow.log_param("n_series_train", len(ok_ids))
        mlflow.log_param("n_series_dropped_too_short", n_dropped)
        model = NBEATS(**model_params)
        nf_run = NeuralForecast(models=[model], freq="W-FRI")
        fit_kwargs = {"df": train_sub[feature_cols]}
        if val_size > 0:
            fit_kwargs["val_size"] = val_size
        nf_run.fit(**fit_kwargs)
        val_fc = nf_run.predict().rename(columns={"NBEATS": "y_pred"})
        val_ev = val_sub.merge(val_fc, on=["unique_id", "ds"], how="inner")
        v_wmae = wmae(val_ev["y"].values, val_ev["y_pred"].values, val_ev["is_holiday_flag"].values.astype(bool))
        train_cv_run = nf_run.cross_validation(
            df=train_sub[feature_cols], n_windows=1, step_size=model_params["h"],
            val_size=val_size,   # ⬅️ ეს არის ერთადერთი დამატებული ხაზი
        ).rename(columns={"NBEATS": "y_pred"})
        train_cv_run = train_cv_run.merge(
            nf_df[["unique_id", "ds", "is_holiday_flag"]], on=["unique_id", "ds"], how="left"
        )
        t_wmae = wmae(
            train_cv_run["y"].values, train_cv_run["y_pred"].values,
            train_cv_run["is_holiday_flag"].values.astype(bool)
        )
        mlflow.log_metric("train_wmae", t_wmae)
        mlflow.log_metric("val_wmae", v_wmae)
        print(f"[{run_name}] train_wmae={t_wmae:,.2f}  val_wmae={v_wmae:,.2f}  "
              f"(series: {len(ok_ids)}, dropped: {n_dropped})")
        return t_wmae, v_wmae, nf_run

In [31]:
nbeats_earlystop_params = dict(
    h=HORIZON, input_size=INPUT_SIZE,
    max_steps=500,               # ჭერი — early stopping უფრო ადრე უნდა ჩერდებოდეს
    val_check_steps=1,
    early_stop_patience_steps=30,
    num_sanity_val_steps=0,
    batch_size=32, learning_rate=1e-3,
    loss=MAE(), random_seed=42, scaler_type="standard",
    devices=1, accelerator="gpu",
)

t_wmae, v_wmae, nf_earlystop = run_pipeline(
    run_name="nbeats_early_stop",
    model_params=nbeats_earlystop_params,
    val_size=HORIZON,   # <-- აუცილებელია early stopping-ისთვის
    notes="early stopping, ავტომატურად პოულობს optimal step-ს baseline-ის ხელით ცდის ნაცვლად",
)
print(f"early_stop val_wmae = {v_wmae:,.2f}")

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
`Trainer(val_check_interval=1)` was configured so validation will run after every batch.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name                | Type          | Params | Mode 
--------------------------------------------------------------
0 | loss                | MAE           | 0      | train
1 | hist_cat_embeddings | ModuleList    | 0      | train
2 | futr_cat_embeddings | ModuleList    | 0      | train
3 | stat_cat_embeddings | ModuleList    | 0      | train
4 | padder_train        | ConstantPad1d | 0      | train
5 | scaler              | TemporalNorm  | 0      | train
6 | blocks              | ModuleList    | 2.5 M  | train
--------------------------------------------------------------
2.5 M     Trainable params
1.0 K     Non-trainable params
2.5 M     Total params
9.978     Total estimated model params size (MB)
34        Modules in train mode
0         Modules in eval m

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting: |          | 0/? [00:00<?, ?it/s]

[nbeats_early_stop] train_wmae=1,432.65  val_wmae=1,384.17  (series: 2944, dropped: 3)
🏃 View run nbeats_early_stop at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/3/runs/aa8dca0ae9ca4949b399c3684c6443ab
🧪 View experiment at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/3
early_stop val_wmae = 1,384.17


In [35]:
import os
import tempfile
import mlflow
import mlflow.pyfunc


class _PatchTSTPyfuncWrapper(mlflow.pyfunc.PythonModel):
    """Wrapper NeuralForecast-ის (PatchTST/NBEATS/ნებისმიერი NF model) native save/load-ის mlflow.pyfunc-თან შესაერთებლად."""

    def load_context(self, context):
        from neuralforecast import NeuralForecast
        self.nf_model = NeuralForecast.load(path=context.artifacts["nf_model_dir"])

    def predict(self, context, model_input=None):
        if model_input is None or (hasattr(model_input, "empty") and model_input.empty):
            return self.nf_model.predict()
        return self.nf_model.predict(df=model_input)


def log_patchtst_pipeline(nf_model, artifact_path: str = "model", registered_model_name: str = None):
    with tempfile.TemporaryDirectory() as tmp_dir:
        save_dir = os.path.join(tmp_dir, "nf_saved_model")
        nf_model.save(path=save_dir, overwrite=True)

        mlflow.pyfunc.log_model(
            artifact_path=artifact_path,
            python_model=_PatchTSTPyfuncWrapper(),
            artifacts={"nf_model_dir": save_dir},
            registered_model_name=registered_model_name,
        )
    print(f"Pipeline დალოგინდა MLflow-ზე, artifact_path='{artifact_path}'")


def load_patchtst_pipeline(model_uri: str):
    return mlflow.pyfunc.load_model(model_uri)

/usr/local/lib/python3.12/dist-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [36]:
with mlflow.start_run(run_name="nbeats_baseline_FINAL_pipeline") as run:
    mlflow.log_params({k: v for k, v in nbeats_baseline_params.items() if k != "loss"})
    mlflow.log_param("loss", type(nbeats_baseline_params["loss"]).__name__)
    mlflow.log_metric("train_wmae", nbeats_train_wmae)
    mlflow.log_metric("val_wmae", nbeats_val_wmae)

    log_patchtst_pipeline(nf_nbeats, artifact_path="model", registered_model_name="NBEATS_best")
    # ⬅️ log_patchtst_pipeline() სახელით მაინც მუშაობს, radgan generic NeuralForecast wrapper-ია
    #    (patch_pipeline.py-ში nf_model.save()/NeuralForecast.load() ნებისმიერ NF model-ს იღებს)

    final_run_id = run.info.run_id
    print("NBEATS pipeline დალოგინდა, run_id:", final_run_id)

2026/07/12 15:06:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 15:06:16 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


2026/07/12 15:06:38 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.25.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.25.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
Successfully registered model 'NBEATS_best'.
2026/07/12 15:06:40 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: NBEATS_best, version 1
Created version '1' of model 'NBEATS_best'.


Pipeline დალოგინდა MLflow-ზე, artifact_path='model'
NBEATS pipeline დალოგინდა, run_id: cf601cff8b184954bc9a7cd4f7d956fd
🏃 View run nbeats_baseline_FINAL_pipeline at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/3/runs/cf601cff8b184954bc9a7cd4f7d956fd
🧪 View experiment at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/3


In [19]:
import os
import tempfile
import mlflow
import mlflow.pyfunc


class _PatchTSTPyfuncWrapper(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        from neuralforecast import NeuralForecast
        self.nf_model = NeuralForecast.load(path=context.artifacts["nf_model_dir"])

    def predict(self, context, model_input=None):
        if model_input is None or (hasattr(model_input, "empty") and model_input.empty):
            return self.nf_model.predict()
        return self.nf_model.predict(df=model_input)


def log_patchtst_pipeline(nf_model, artifact_path: str = "model", registered_model_name: str = None):
    with tempfile.TemporaryDirectory() as tmp_dir:
        save_dir = os.path.join(tmp_dir, "nf_saved_model")
        nf_model.save(path=save_dir, overwrite=True)

        mlflow.pyfunc.log_model(
            artifact_path=artifact_path,
            python_model=_PatchTSTPyfuncWrapper(),
            artifacts={"nf_model_dir": save_dir},
            registered_model_name=registered_model_name,
        )
    print(f"Pipeline დალოგინდა MLflow-ზე, artifact_path='{artifact_path}'")


def load_patchtst_pipeline(model_uri: str):
    return mlflow.pyfunc.load_model(model_uri)

/usr/local/lib/python3.12/dist-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [20]:
n_test_weeks = 39  # ადრე დათვლილი test.csv-დან

full_train_df = pd.concat([train_df, val_df], ignore_index=True).sort_values(["unique_id", "ds"])

nbeats_final_config = dict(
    h=n_test_weeks,           # 8 → 39
    input_size=INPUT_SIZE,    # იგივე baseline-ის, 52
    max_steps=200,            # იგივე baseline-ის საუკეთესო
    batch_size=32,
    learning_rate=1e-3,
    loss=MAE(), random_seed=42, scaler_type="standard",
    devices=1, accelerator="gpu",
)

with mlflow.start_run(run_name="nbeats_FINAL_submission_model") as run:
    mlflow.log_params({k: v for k, v in nbeats_final_config.items() if k != "loss"})
    mlflow.log_param("loss", type(nbeats_final_config["loss"]).__name__)
    mlflow.log_param("trained_on", "train_df + val_df combined (full history)")
    mlflow.log_param("horizon_weeks", n_test_weeks)

    final_nbeats_model = NBEATS(**nbeats_final_config)
    nf_final_nbeats = NeuralForecast(models=[final_nbeats_model], freq="W-FRI")
    nf_final_nbeats.fit(df=full_train_df[["unique_id", "ds", "y"]])

    log_patchtst_pipeline(nf_final_nbeats, artifact_path="model", registered_model_name="NBEATS_final_submission")

    final_nbeats_run_id = run.info.run_id
    print("საბოლოო NBEATS model დალოგინდა, run_id:", final_nbeats_run_id)

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name                | Type          | Params | Mode 
--------------------------------------------------------------
0 | loss                | MAE           | 0      | train
1 | hist_cat_embeddings | ModuleList    | 0      | train
2 | futr_cat_embeddings | ModuleList    | 0      | train
3 | stat_cat_embeddings | ModuleList    | 0      | train
4 | padder_train        | ConstantPad1d | 0      | train
5 | scaler              | TemporalNorm  | 0      | train
6 | blocks              | ModuleList    | 2.6 M  | train
--------------------------------------------------------------
2.6 M     Trainable params
7.2 K     Non-trainable params
2.6 M     Total params
10.321    Total estimated model params size (MB)
34        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=200` reached.
2026/07/12 15:44:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 15:44:02 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


2026/07/12 15:44:23 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.25.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.25.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
Successfully registered model 'NBEATS_final_submission'.
2026/07/12 15:44:27 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: NBEATS_final_submission, version 1
Created version '1' of model 'NBEATS_final_submission'.


Pipeline დალოგინდა MLflow-ზე, artifact_path='model'
საბოლოო NBEATS model დალოგინდა, run_id: 5e0c4f550001410b9597622a3c9d2919
🏃 View run nbeats_FINAL_submission_model at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/3/runs/5e0c4f550001410b9597622a3c9d2919
🧪 View experiment at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/3


In [23]:
predictions = nf_final_nbeats.predict()

predictions[["Store", "Dept"]] = predictions["unique_id"].str.split("_", expand=True)

submission = predictions.copy()
submission["Id"] = (
    submission["Store"].astype(str) + "_" +
    submission["Dept"].astype(str) + "_" +
    pd.to_datetime(submission["ds"]).dt.strftime("%Y-%m-%d")
)

pred_col = "NBEATS" if "NBEATS" in submission.columns else "y_pred"
submission = submission[["Id", pred_col]].rename(columns={pred_col: "Weekly_Sales"})

sample_sub = pd.read_csv("/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/sampleSubmission.csv.zip")
print("sample:", len(sample_sub), "ჩვენი:", len(submission))
print("Id-ები ემთხვევა?", set(submission["Id"]) == set(sample_sub["Id"]))
print("NaN-ები:", submission["Weekly_Sales"].isna().sum())

submission.to_csv("nbeats_submission.csv", index=False)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

sample: 115064 ჩვენი: 114933
Id-ები ემთხვევა? False
NaN-ები: 0


In [25]:
sample_sub = pd.read_csv("/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/sampleSubmission.csv.zip")

missing_ids = set(sample_sub["Id"]) - set(submission["Id"])
extra_ids = set(submission["Id"]) - set(sample_sub["Id"])

print("სულ ნაკლული Id:", len(missing_ids))
print("სულ ზედმეტი Id:", len(extra_ids))
print("\nმაგალითები ნაკლული Id-ებიდან:")
print(list(missing_ids)[:10])

სულ ნაკლული Id: 2437
სულ ზედმეტი Id: 2306

მაგალითები ნაკლული Id-ებიდან:
['39_19_2013-06-07', '35_49_2012-11-16', '30_33_2013-05-03', '16_93_2013-06-07', '42_26_2013-05-10', '13_99_2012-12-14', '3_49_2013-05-31', '27_47_2013-02-22', '37_22_2013-04-12', '11_99_2013-05-24']


In [26]:
missing_df = pd.DataFrame({"Id": list(missing_ids)})
missing_df[["Store", "Dept", "Date"]] = missing_df["Id"].str.rsplit("_", n=2, expand=True)  # ბოლო 2 "_"-ით გავყოთ, radgan Date-შიც არის "-", magram "_" არ არის, ასე რომ ჩვეულებრივი split საკმარისია
print("უნიკალური Store-Dept წყვილები, რომლებიც აკლია:")
print(missing_df[["Store", "Dept"]].drop_duplicates())

# ეს წყვილები არსებობდა თუ არა train_df-ში?
missing_pairs = missing_df[["Store", "Dept"]].drop_duplicates()
missing_pairs["unique_id"] = missing_pairs["Store"].astype(str) + "_" + missing_pairs["Dept"].astype(str)
in_train = missing_pairs["unique_id"].isin(full_train_df["unique_id"].unique())
print("ამ წყვილებიდან რამდენი იყო train-ში?", in_train.sum(), "/", len(missing_pairs))

უნიკალური Store-Dept წყვილები, რომლებიც აკლია:
     Store Dept
0       39   19
1       35   49
2       30   33
3       16   93
4       42   26
...    ...  ...
2335     9   54
2366     8   47
2380    37   71
2410     6   47
2423    38   20

[242 rows x 2 columns]
ამ წყვილებიდან რამდენი იყო train-ში? 19 / 242


In [27]:
# Fallback: აკლი Store-Dept წყვილებისთვის, გამოვიყენოთ იმავე Dept-ის საშუალო sales
# (ან, თუ Dept-იც ახალია, მთლიანი გლობალური საშუალო)
dept_avg = full_train_df.groupby(full_train_df["unique_id"].str.split("_").str[1])["y"].mean()

fallback_rows = []
for _, row in missing_df.iterrows():
    dept = row["Dept"]
    fallback_val = dept_avg.get(dept, full_train_df["y"].mean())
    fallback_rows.append({"Id": row["Id"], "Weekly_Sales": fallback_val})

fallback_df = pd.DataFrame(fallback_rows)
submission_complete = pd.concat([submission, fallback_df], ignore_index=True)

print("საბოლოო submission row count:", len(submission_complete))
print("სრულად ემთხვევა sample-ს?", set(submission_complete["Id"]) == set(sample_sub["Id"]))

submission_complete.to_csv("nbeats_submission_complete.csv", index=False)

საბოლოო submission row count: 117370
სრულად ემთხვევა sample-ს? False


In [30]:
# 1. სრული test grid, Kaggle-ის sampleSubmission-იდან
sample_sub = pd.read_csv("/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/sampleSubmission.csv.zip")
sample_sub[["Store", "Dept", "Date"]] = sample_sub["Id"].str.split("_", n=2, expand=True)
sample_sub["Date"] = pd.to_datetime(sample_sub["Date"])
sample_sub["unique_id"] = sample_sub["Store"] + "_" + sample_sub["Dept"]

# 2. prediction-ის შედეგი unique_id/ds-ზე merge, LEFT join რომ ყველა საჭირო row შემორჩეს
pred_col = "NBEATS" if "NBEATS" in predictions.columns else "y_pred"
merged = sample_sub.merge(
    predictions[["unique_id", "ds", pred_col]],
    left_on=["unique_id", "Date"], right_on=["unique_id", "ds"],
    how="left"
)

print("NaN გახდა (prediction ვერ მოიძებნა):", merged[pred_col].isna().sum())

# 3. Fallback - Dept-ის საშუალო, ან სერიის საკუთარი ისტორიული საშუალო
dept_avg = full_train_df.groupby(full_train_df["unique_id"].str.split("_").str[1])["y"].mean()
series_avg = full_train_df.groupby("unique_id")["y"].mean()

def fallback_value(row):
    if pd.notna(row[pred_col]):
        return row[pred_col]
    if row["unique_id"] in series_avg.index:
        return series_avg[row["unique_id"]]
    return dept_avg.get(row["Dept"], full_train_df["y"].mean())

merged["Weekly_Sales"] = merged.apply(fallback_value, axis=1)

submission_final = merged[["Id", "Weekly_Sales"]]
print("საბოლოო row count:", len(submission_final), "/ sample:", len(sample_sub))
print("Id-ები ზუსტად ემთხვევა?", set(submission_final["Id"]) == set(sample_sub["Id"]))
print("NaN-ები:", submission_final["Weekly_Sales"].isna().sum())

submission_final.to_csv("nbeats_submission_complete.csv", index=False)

NaN გახდა (prediction ვერ მოიძებნა): 2437
საბოლოო row count: 115064 / sample: 115064
Id-ები ზუსტად ემთხვევა? True
NaN-ები: 0
